# Envelope-detector GR metrics — LA2A (SignalTrain, whole `all/` set)

LA2A companion to [envelope_metrics_diffssl.ipynb](envelope_metrics_diffssl.ipynb). Same pipeline,
same metric engine, same metric set — only the dataset and the dry/wet pairing differ. It **quantifies**
every envelope / level detector over the **entire SignalTrain LA-2A set** (`data/LA2A/all`, all
source audios × all Comp/Limit × Peak-Reduction settings), reporting **overall, duration-weighted**
metrics only (no per-pair rows).

For each detector the pipeline is identical to the diff-SSL notebook:
`envelope → dB → GR → gain → reconstruction`, with the metrics grouped into two stages plus the
standard set used in the SOTA eval notebooks.

**Detectors:** Hilbert · RMS-64 · RMS-256 · RMS-1024 · RMS-4096 ·
**AR 64/1024** (asymmetric causal one-pole — fast attack τ≈1.5 ms, slow release τ≈23 ms) ·
Simple diff (per-sample `wet/dry`, the degenerate near-perfect-reconstruction anchor).

### Gain-reduction stage — the slow GR trajectory standard losses under-fit
| Metric | Measures | Source |
|---|---|---|
| **GR MAE (dB)** | mean \|GR_est − GR_ref\| on the trajectory | eval notebook |
| **MR-STE** | multi-resolution short-time **energy** envelope distance — shape, tolerant of small misalignments | Wright & Välimäki (grey-box) |

### Colour stage — residual timbre the smooth gain model can't explain
| Metric | Measures | Source |
|---|---|---|
| **MR-STFT (auraloss)** | multi-resolution STFT distance (spectral-convergence + log-magnitude) | auraloss, via `nablafx.evaluation` |
| **ESR (A-wt)** | A-weighted error-to-signal ratio — paper-comparable headline number | Wright & Välimäki |

### Standard (same as the SOTA eval notebook)
MAE (L1) · MSE (L2) · EDC · M_NRMSE · M_SF.

**LA2A specifics**
- Pairing comes from `gr_dataset.discover_la2a_pairs`: each `target_<n>_LA2A_<c>c__<comp/limit>__<PR>.wav`
  is paired with its `input_<n>_.wav`. A **"sound"** is a distinct source audio (the `<c>c` chunk —
  same input audio re-used across all of that chunk's settings); a **"setting"** is a
  `(Comp/Limit, Peak-Reduction)` combo.
- 44.1 kHz mono; the streaming reader truncates each pair to the shorter of dry/wet (handles the
  SignalTrain time-alignment fixes noted in `data/LA2A/changes.txt`).
- Heavy metrics run through the same **batched, cumsum-accelerated** engine as the diff-SSL notebook
  (the AR detector's per-sample recursion is numba-jitted so it adds negligible cost). The verification
  cell asserts the colour-stage MR-STFT / ESR reproduce `nablafx.evaluation` (auraloss) to < 1e-3.
  Set `MAX_EVAL_PAIRS` to a small int for a quick smoke test.
- **Runtime note:** LA2A holds ~24 h of processed audio (vs ~7 h for diff-SSL), so a full run is
  ~3–4× longer than the diff-SSL one (≈ 3–4 h). The stats and verification cells below are cheap and
  run in seconds.

In [1]:
import os
import sys
import gc
import types
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from scipy.signal import hilbert, bilinear, lfilter
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# Resolve repo root robustly (independent of the launch directory)
REPO_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file()),
    Path.cwd().resolve(),
)
# eval_helpers (segment reader) + gr_dataset (LA2A pairing) live in 03_initial_GR_pred,
# the same source the SOTA eval notebooks use.
sys.path.insert(0, str(REPO_ROOT / "03_initial_GR_pred"))

# nablafx's top-level __init__ eagerly imports processors.ddsp -> `rational`
# (an optional dependency that isn't installed here). Stub it so the pure
# auraloss/torch wrappers in `nablafx.evaluation` import cleanly.
for _name in ("rational", "rational.torch"):
    sys.modules.setdefault(_name, types.ModuleType(_name))
sys.modules["rational.torch"].Rational = object

from eval_helpers import _read_dry_wet_segment, _pair_num_frames
from gr_dataset import discover_la2a_pairs
from src.dsp import to_amplitude
from src.dsp_torch import GR_DB_MIN, GR_DB_MAX, RMS_WINDOW
from nablafx.evaluation import get_function, list_available_metrics

print(f"repo : {REPO_ROOT}")
print(f"torch: {torch.__version__}")
print(f"nablafx.evaluation metrics: {len(list_available_metrics())} registered "
      f"(using mrstft_loss, esr_loss)")

repo : /Volumes/Saola's Drive/AllCode/thesis/Virtual-Analogue-Compressor-Modelling
torch: 2.10.0
nablafx.evaluation metrics: 25 registered (using mrstft_loss, esr_loss)


## Configuration

In [ ]:
# LA2A (SignalTrain Teletronix LA-2A hardware, 44.1 kHz mono)
DATA_PARENT = "/Volumes/Saola's Drive/AllCode/thesis/data"   # folder that contains LA2A/
DATA_ROOT   = os.path.join(DATA_PARENT, "LA2A")
SAMPLE_RATE = 44100
EPS         = 1e-10

# Streaming chunking -> bounded memory; STEP alignment matches the SOTA eval notebooks
STEP             = 256 * 8                       # 2048
STREAM_CHUNK_SEC = 10.0
CHUNK_FRAMES     = int(round(STREAM_CHUNK_SEC * SAMPLE_RATE))
CHUNK_FRAMES    -= CHUNK_FRAMES % STEP

# Envelope detectors compared (exactly those in envelope_gr_comparison.ipynb)
RMS_WINDOWS    = (64, 256, 1024, 4096)
AR_ATT, AR_REL = 64, 1024                        # asymmetric one-pole attack/release [samples]
AR_LABEL       = f"AR {AR_ATT}/{AR_REL}"          # fast attack (RMS-64-like) + slow release (RMS-1024-like)
DETECTORS      = ["Hilbert"] + [f"RMS {w}" for w in RMS_WINDOWS] + [AR_LABEL, "Simple diff"]

# Multi-resolution window / FFT sets
MR_STE_WINDOWS   = (256, 1024, 4096)            # short-time energy (envelope shape)
MR_NRMSE_WINDOWS = (512, 1024, 2048)            # == src.losses.multi_resolution_nrmse
STFT_SIZES       = (512, 1024, 2048)            # == src.losses.multi_resolution_spectral_flux_error

# Quick smoke test: set to a small int (e.g. 2) to run on a few pairs only
MAX_EVAL_PAIRS = None

OUT_CSV = REPO_ROOT / "01b_dset_anal" / "envelope_metrics_la2a.csv"

# Metric columns, grouped: GR-stage | colour-stage | standard (eval-notebook)
GR_STAGE_COLS     = ["GR MAE (dB)", "MR-STE"]
COLOUR_STAGE_COLS = ["MR-STFT (auraloss)", "ESR (A-wt)"]
STANDARD_COLS     = ["MAE (L1)", "MSE (L2)", "EDC", "M_NRMSE", "M_SF"]
METRIC_COLS       = GR_STAGE_COLS + COLOUR_STAGE_COLS + STANDARD_COLS

assert os.path.isdir(DATA_ROOT), f"Missing DATA_ROOT: {DATA_ROOT}"
print(f"chunk    : {STREAM_CHUNK_SEC}s ~ {CHUNK_FRAMES} frames, step={STEP}")
print(f"detectors: {DETECTORS}")
print(f"metrics  : {METRIC_COLS}")

## Metric engine

All moving-average (RMS / energy) envelopes use a **prefix-sum (cumsum)** sliding window — `O(N)`
regardless of window length, validated to match a direct `conv1d` to ~1e-7. The FFT-based metrics are
computed **once per chunk, batched across all detectors** (the target STFT is shared), which is what
keeps the run in bounded memory. Identical to the diff-SSL notebook's engine.

In [ ]:
# ── Envelope / level helpers (cumsum moving-average: O(N), window-independent) ──
from numba import njit


def _moving_meansq(x_sq: np.ndarray, W: int, centered: bool) -> np.ndarray:
    """Sliding mean of x**2 via prefix sums. centered=symmetric window, else causal (trailing)."""
    n = x_sq.shape[-1]
    c = np.empty(n + 1); c[0] = 0.0; np.cumsum(x_sq, out=c[1:])
    i = np.arange(n)
    if centered:
        lo = i - (W // 2); hi = lo + W
    else:                                       # causal window == src.dsp_torch.gain_reduction_db
        hi = i + 1; lo = hi - W
    lo = np.clip(lo, 0, n); hi = np.clip(hi, 0, n)
    return (c[hi] - c[lo]) / np.maximum(hi - lo, 1)

def _env_rms(x, W): return np.sqrt(_moving_meansq(x * x, W, centered=True))
def _to_db(x):      return 20.0 * np.log10(np.maximum(x, EPS))
def _esr(pred, tgt): return float(np.sum((tgt - pred) ** 2) / (np.sum(tgt * tgt) + 1e-8))

@njit
def _env_ar_onepole(x, att, rel):
    """Causal asymmetric one-pole RMS (== env_ar_onepole in envelope_gr_comparison.ipynb):
    smooth x**2 with a fast time constant (att) while the power rises and a slow one (rel)
    while it falls, then sqrt. numba-jit so the per-sample recursion stays fast over the
    whole streamed dataset."""
    a_att = np.exp(-1.0 / att)
    a_rel = np.exp(-1.0 / rel)
    n = x.shape[0]
    y = np.empty(n)
    s = x[0] * x[0]
    for i in range(n):
        pn = x[i] * x[i]
        a = a_att if pn > s else a_rel          # fast while rising, slow while falling
        s = a * s + (1.0 - a) * pn
        y[i] = s
    return np.sqrt(y)

# A-weighting IIR (analog prototype -> bilinear) for the A-weighted ESR
def _a_weighting_ba(fs):
    f1, f2, f3, f4 = 20.598997, 107.65265, 737.86223, 12194.217
    A1000 = 1.9997
    nums = [(2 * np.pi * f4) ** 2 * 10 ** (A1000 / 20.0), 0, 0, 0, 0]
    dens = np.polymul([1, 4 * np.pi * f4, (2 * np.pi * f4) ** 2],
                      [1, 4 * np.pi * f1, (2 * np.pi * f1) ** 2])
    dens = np.polymul(np.polymul(dens, [1, 2 * np.pi * f3]), [1, 2 * np.pi * f2])
    return bilinear(nums, dens, fs)
AW_B, AW_A = _a_weighting_ba(SAMPLE_RATE)

# ── GR trajectories ──
def detector_gr_db(label, dry, wet):
    """Estimated gain-reduction trajectory (dB) for one envelope detector (unclamped)."""
    if label == "Hilbert":
        return _to_db(np.abs(hilbert(wet))) - _to_db(np.abs(hilbert(dry)))
    if label.startswith("RMS"):
        W = int(label.split()[1])
        return _to_db(_env_rms(wet, W)) - _to_db(_env_rms(dry, W))
    if label.startswith("AR"):                  # asymmetric causal one-pole (fast attack / slow release)
        return (_to_db(_env_ar_onepole(wet, AR_ATT, AR_REL))
                - _to_db(_env_ar_onepole(dry, AR_ATT, AR_REL)))
    # Simple diff: degenerate per-sample gain wet/dry (perfect recon, noisy GR)
    gain = np.divide(wet, dry, out=np.ones_like(wet), where=np.abs(dry) > EPS)
    return _to_db(np.abs(gain))

def reference_gr_db(dry, wet, W=RMS_WINDOW):
    """Reference GR trajectory = causal RMS GR (matches src.dsp_torch.gain_reduction_db)."""
    return (_to_db(np.sqrt(_moving_meansq(wet * wet, W, centered=False)))
            - _to_db(np.sqrt(_moving_meansq(dry * dry, W, centered=False))))

# ── GR-stage metrics ──
def mr_ste(pred, wet, windows=MR_STE_WINDOWS):
    """Multi-resolution short-time energy distance (envelope shape; Wright & Valimaki)."""
    pe, we = pred * pred, wet * wet
    total = 0.0
    for W in windows:
        ep = _moving_meansq(pe, W, centered=True)
        et = _moving_meansq(we, W, centered=True)
        total += np.sum(np.abs(ep - et)) / (np.sum(np.abs(et)) + 1e-8)
    return float(total / len(windows))

def mr_nrmse(pred, wet, windows=MR_NRMSE_WINDOWS):
    """Mean normalised RMS-envelope error (== src.losses.multi_resolution_nrmse, cumsum-fast)."""
    total = 0.0
    for W in windows:
        rp = np.sqrt(_moving_meansq(pred * pred, W, centered=False))
        rt = np.sqrt(_moving_meansq(wet * wet, W, centered=False))
        total += np.sqrt(np.mean((rt - rp) ** 2)) / (np.sqrt(np.mean(rt * rt)) + 1e-8)
    return float(total / len(windows))

In [ ]:
# ── Colour-stage + standard FFT metrics, batched across all detectors at once ──
_MRSTFT_RES = [(1024, 120, 600), (2048, 240, 1200), (512, 50, 240)]   # auraloss MRSTFT defaults
_HANN = {}
def _win(n):
    if n not in _HANN:
        _HANN[n] = torch.hann_window(n)
    return _HANN[n]

def _stft_mag_pow(x2d, n_fft, hop, win):
    """auraloss-style magnitude = sqrt(clamp(re^2 + im^2, 1e-8))."""
    st = torch.stft(x2d, n_fft=n_fft, hop_length=hop, win_length=win,
                    window=_win(win), return_complex=True)
    return torch.sqrt(torch.clamp(st.real ** 2 + st.imag ** 2, min=1e-8))

def _stft_mag(x2d, n_fft):
    """src.losses-style magnitude (hop = n_fft // 4, hann(n_fft))."""
    return torch.stft(x2d, n_fft=n_fft, hop_length=n_fft // 4,
                      window=_win(n_fft), return_complex=True).abs()

@torch.no_grad()
def fft_metrics_batched(preds2d: torch.Tensor, wet1d: torch.Tensor) -> dict:
    """preds2d: [D, T] (one row per detector), wet1d: [T]. Returns {col: list-of-D-floats}.

    Per-item reimplementations of auraloss MultiResolutionSTFTLoss and of
    src.losses.{multi_resolution_spectral_flux_error, edc}, validated to match the
    originals to <1e-3 / ~1e-7 respectively.
    """
    D = preds2d.shape[0]
    out = {}

    # MR-STFT (auraloss default = spectral-convergence + log-magnitude, mean over 3 resolutions)
    acc = torch.zeros(D)
    for n_fft, hop, win in _MRSTFT_RES:
        P = _stft_mag_pow(preds2d, n_fft, hop, win)
        T = _stft_mag_pow(wet1d.unsqueeze(0), n_fft, hop, win)[0]
        sc = torch.linalg.norm((T.unsqueeze(0) - P).reshape(D, -1), dim=1) / torch.linalg.norm(T.reshape(-1))
        lm = (torch.log(P) - torch.log(T.unsqueeze(0))).abs().mean(dim=(1, 2))
        acc += sc + lm
    out["MR-STFT (auraloss)"] = (acc / len(_MRSTFT_RES)).tolist()

    # src.losses spectral-flux (M_SF) over (512, 1024, 2048)
    sf = torch.zeros(D)
    for n_fft in STFT_SIZES:
        P = _stft_mag(preds2d, n_fft)
        T = _stft_mag(wet1d.unsqueeze(0), n_fft)[0]
        pf = torch.diff(P, dim=-1); tf = torch.diff(T, dim=-1)
        sf += (tf.unsqueeze(0) - pf).abs().mean(dim=(1, 2)) / (tf.abs().mean() + 1e-8)
    out["M_SF"] = (sf / len(STFT_SIZES)).tolist()

    # EDC (energy-decay-curve error, dB)
    p = preds2d; t = wet1d.unsqueeze(0)
    pe = torch.flip(torch.cumsum(torch.flip(p ** 2, dims=(-1,)), dim=-1), dims=(-1,))
    te = torch.flip(torch.cumsum(torch.flip(t ** 2, dims=(-1,)), dim=-1), dims=(-1,))
    pe = pe / (pe[..., :1] + 1e-8); te = te / (te[..., :1] + 1e-8)
    out["EDC"] = ((10 * torch.log10(te.clamp(min=1e-8)) - 10 * torch.log10(pe.clamp(min=1e-8)))
                  .abs().mean(dim=-1)).tolist()
    return out

In [ ]:
# ── Per-chunk and per-pair drivers ──
def chunk_metrics(dry: np.ndarray, wet: np.ndarray) -> dict:
    """All metrics for one chunk, every detector. -> {label: {col: float}}."""
    gr_tgt = np.clip(reference_gr_db(dry, wet), GR_DB_MIN, GR_DB_MAX)
    wet_aw = lfilter(AW_B, AW_A, wet)

    preds, rows = [], {}
    for label in DETECTORS:
        gr = detector_gr_db(label, dry, wet)
        pred = dry * to_amplitude(gr)                       # reconstruction (unclamped GR)
        preds.append(pred)
        gr_c = np.clip(gr, GR_DB_MIN, GR_DB_MAX)
        rows[label] = {
            "GR MAE (dB)": float(np.mean(np.abs(gr_c - gr_tgt))),
            "MR-STE":      mr_ste(pred, wet),
            "ESR (A-wt)":  _esr(lfilter(AW_B, AW_A, pred), wet_aw),
            "MAE (L1)":    float(np.mean(np.abs(pred - wet))),
            "MSE (L2)":    float(np.mean((pred - wet) ** 2)),
            "M_NRMSE":     mr_nrmse(pred, wet),
        }

    fm = fft_metrics_batched(torch.from_numpy(np.stack(preds)).float(),
                             torch.from_numpy(wet).float())
    for i, label in enumerate(DETECTORS):
        for col in ("MR-STFT (auraloss)", "M_SF", "EDC"):
            rows[label][col] = fm[col][i]
    return rows

def stream_pair(dry_p, wet_p):
    """Frame-weighted metric sums for one (dry, wet) LA2A pair (streamed, bounded memory)."""
    total = _pair_num_frames(dry_p, wet_p, SAMPLE_RATE)
    acc = {l: {c: 0.0 for c in METRIC_COLS} for l in DETECTORS}
    n_frames = 0
    for o in range(0, total, CHUNK_FRAMES):
        d_t, w_t = _read_dry_wet_segment(dry_p, wet_p, o, min(o + CHUNK_FRAMES, total), SAMPLE_RATE)
        L = d_t.shape[-1] - (d_t.shape[-1] % STEP)
        if L < STEP:
            break
        dry = d_t.squeeze(0).numpy().astype(np.float64)[:L]
        wet = w_t.squeeze(0).numpy().astype(np.float64)[:L]
        cm = chunk_metrics(dry, wet)
        for l in DETECTORS:
            for c in METRIC_COLS:
                acc[l][c] += cm[l][c] * L
        n_frames += L
        del d_t, w_t, dry, wet, cm
    return acc, n_frames

## Discover every (sound, setting) pair in `data/LA2A/all`

`discover_la2a_pairs` matches each `target_<n>_LA2A_<c>c__<comp/limit>__<PR>.wav` to its
`input_<n>_.wav`. We tag each pair with a **sound** (the `<c>c` chunk = distinct source audio) and a
**setting** (`cl<comp/limit>_pr<PR>`).

In [6]:
PAIRS = discover_la2a_pairs(DATA_PARENT)
for p in PAIRS:
    p["sound"]   = p["channel_config"]                               # e.g. '2c', '3c'
    p["setting"] = f"cl{p['comp_limit']}_pr{p['peak_reduction']:03d}"
    p["label"]   = f"{p['sound']}_{p['setting']}"
PAIRS = sorted(PAIRS, key=lambda p: (p["sound"], p["comp_limit"], p["peak_reduction"]))

sounds   = sorted({p["sound"] for p in PAIRS})
settings = sorted({p["setting"] for p in PAIRS})
print(f"sounds: {len(sounds)} {sounds}   settings: {len(settings)}   (sound, setting) pairs: {len(PAIRS)}")

sounds: 2 ['2c', '3c']   settings: 42   (sound, setting) pairs: 84


## Dataset stats

Cheap (header-only via `soundfile.info`, no decode). Reports the overall length of the set, the
control-parameter space, and the number of distinct sounds — the LA2A counterparts of the diff-SSL
`settings`/`songs`/duration figures.

- A **sound** is a distinct source audio (`<c>c` chunk). Within a chunk the *same* input is re-used
  across every setting, so "unique source material" counts each chunk's input once, while
  "total processed audio" counts every pair (the figure comparable to the diff-SSL run's reported minutes).

In [7]:
import soundfile as sf

def _dur_min(path):
    info = sf.info(path)
    return info.frames / info.samplerate / 60.0

by_sound = defaultdict(list)
for p in PAIRS:
    by_sound[p["sound"]].append(p)

rows, total_processed = [], 0.0
for s in sorted(by_sound):
    plist = by_sound[s]
    src_min  = _dur_min(plist[0]["dry_path"])               # one representative input = source length
    proc_min = sum(_dur_min(p["wet_path"]) for p in plist)  # every pair (counts the re-used source)
    total_processed += proc_min
    rows.append({
        "sound": s,
        "n_settings":     len({p["setting"] for p in plist}),
        "n_pairs":        len(plist),
        "source_len_min": round(src_min, 2),
        "processed_min":  round(proc_min, 1),
    })
stats_df = pd.DataFrame(rows)

cl_vals = sorted({p["comp_limit"] for p in PAIRS})
pr_vals = sorted({p["peak_reduction"] for p in PAIRS})
unique_src = stats_df["source_len_min"].sum()

print("LA2A (SignalTrain) — dataset stats")
print(f"  distinct sounds (source audios)  : {len(by_sound)}   {sorted(by_sound)}")
print(f"  control params (knobs)           : 2")
print(f"      - Comp/Limit                 : {cl_vals}   (0 = Comp, 1 = Limit)")
print(f"      - Peak Reduction             : {pr_vals[0]}..{pr_vals[-1]} step 5   ({len(pr_vals)} values)")
print(f"  distinct settings (Comp/Limit x PR): {len(settings)}")
print(f"  (sound, setting) pairs           : {len(PAIRS)}")
print(f"  unique source material           : {unique_src:.1f} min  ({unique_src/60:.2f} h)")
print(f"  total processed audio (all pairs): {total_processed:.1f} min  ({total_processed/60:.2f} h)")
stats_df

LA2A (SignalTrain) — dataset stats
  distinct sounds (source audios)  : 2   ['2c', '3c']
  control params (knobs)           : 2
      - Comp/Limit                 : [0, 1]   (0 = Comp, 1 = Limit)
      - Peak Reduction             : 0..100 step 5   (21 values)
  distinct settings (Comp/Limit x PR): 42
  (sound, setting) pairs           : 84
  unique source material           : 35.0 min  (0.58 h)
  total processed audio (all pairs): 1459.0 min  (24.32 h)


  sound  n_settings  n_pairs  source_len_min  processed_min
0    2c          41       41            15.0          615.0
1    3c          42       43            20.0          844.0

## Verify the fast engine reproduces `nablafx` / auraloss

On the first chunk of the first pair, compare the batched colour-stage MR-STFT and ESR against
`nablafx.evaluation.get_function("mrstft_loss")` (auraloss `MultiResolutionSTFTLoss`) and
`get_function("esr_loss")` (auraloss `ESRLoss`) called per detector.

In [8]:
_p0 = PAIRS[0]
_dp, _wp = _p0["dry_path"], _p0["wet_path"]
_d, _w = _read_dry_wet_segment(_dp, _wp, 0, CHUNK_FRAMES, SAMPLE_RATE)
_L = _d.shape[-1] - (_d.shape[-1] % STEP)
_dry = _d.squeeze(0).numpy().astype(np.float64)[:_L]
_wet = _w.squeeze(0).numpy().astype(np.float64)[:_L]

_preds = np.stack([_dry * to_amplitude(detector_gr_db(l, _dry, _wet)) for l in DETECTORS])
_fm = fft_metrics_batched(torch.from_numpy(_preds).float(), torch.from_numpy(_wet).float())

_nab_mrstft = get_function("mrstft_loss")     # auraloss MultiResolutionSTFTLoss
_nab_esr    = get_function("esr_loss")        # auraloss ESRLoss
_wt = torch.from_numpy(_wet).float().view(1, 1, -1)

print(f"verifying on {_p0['label']}  (chunk 0)\n")
print(f"{'detector':12s} {'MRSTFT batched':>15s} {'MRSTFT nablafx':>15s} "
      f"{'ESR batched':>12s} {'ESR nablafx':>12s}")
for i, l in enumerate(DETECTORS):
    pt = torch.from_numpy(_preds[i]).float().view(1, 1, -1)
    nm, ne = float(_nab_mrstft(pt, _wt)), float(_nab_esr(pt, _wt))
    bm, be = _fm["MR-STFT (auraloss)"][i], _esr(_preds[i], _wet)
    assert abs(nm - bm) < 1e-3, (l, nm, bm)
    assert abs(ne - be) < 1e-4, (l, ne, be)
    print(f"{l:12s} {bm:15.6f} {nm:15.6f} {be:12.6f} {ne:12.6f}")
print("\nOK — batched fast path matches nablafx / auraloss.")

verifying on 2c_cl0_pr000  (chunk 0)

detector      MRSTFT batched  MRSTFT nablafx  ESR batched  ESR nablafx
Hilbert             0.666848        0.666850     0.000341     0.000341
RMS 64              0.781402        0.781403     0.000481     0.000481
RMS 256             0.777401        0.777402     0.000595     0.000595
RMS 1024            0.776587        0.776588     0.000591     0.000591
RMS 4096            0.776524        0.776525     0.000592     0.000592
Simple diff         1.252608        1.252608     0.000013     0.000013

OK — batched fast path matches nablafx / auraloss.


## Run over the whole dataset

Overall, **duration-weighted** metrics only. LA2A holds ~24 h of processed audio, so a full run is
≈ 3–4 h (progress bar below). Lower the iteration count with `MAX_EVAL_PAIRS` for a smoke test.

In [ ]:
pairs_to_eval = PAIRS if MAX_EVAL_PAIRS is None else PAIRS[:MAX_EVAL_PAIRS]

overall = {l: {c: 0.0 for c in METRIC_COLS} for l in DETECTORS}
total_frames = 0
for pair in tqdm(pairs_to_eval, desc="pairs", unit="pair"):
    acc, n = stream_pair(pair["dry_path"], pair["wet_path"])
    for l in DETECTORS:
        for c in METRIC_COLS:
            overall[l][c] += acc[l][c]
    total_frames += n
    gc.collect()

print(f"\npairs evaluated: {len(pairs_to_eval)}   audio: {total_frames / SAMPLE_RATE / 60:.1f} min")

## Overall results

In [ ]:
overall_df = pd.DataFrame(
    [{"Detector": l, **{c: overall[l][c] / total_frames for c in METRIC_COLS}} for l in DETECTORS]
)
overall_df.to_csv(OUT_CSV, index=False)
print(f"saved -> {OUT_CSV}")

# Grouped view: GR-stage | colour-stage | standard
overall_df.set_index("Detector")[METRIC_COLS].round(6)